**SETUP**


In [2]:
import pandas as pd
import numpy as np
print("Working")

Working


In [3]:
pd.set_option("display.max_columns",50)


**LOAD THE DATASET**
(TITANIC.CSV)

In [4]:
df=pd.read_csv("C:/Users/beast/OneDrive/Desktop/shared-files/Python/MLcourse/EDA/titanic.csv")
df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [6]:
df.shape

(891, 12)

ADDING FEW COLUMNS FOR DEMONSTRATING DATA ISSUES


In [ ]:
df["constant_col"]=1  #doesnt have any relatonship with the output survived
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,constant_col
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1


**CREATING A DIRTY VERSION OF ANY COLUMN WITH TEXT MISMATCH OR SOMETHING WITH ERROR**

In [11]:
df["Name_dirty"]=df["Name"].copy()
#select  some non missing values rows
sample_idx=df["Name_dirty"].dropna().sample(6,random_state=42).index # name_dirty column se hatao missing values waale baakiyo mai se s
                                                                       #select 6 random jinme values hongi  

#apply simple transformation 
df.loc[sample_idx[0:2],"Name_dirty"]=df.loc[sample_idx[0:2],"Name_dirty"].str.upper()
df.loc[sample_idx[2:4],"Name_dirty"]=df.loc[sample_idx[0:2],"Name_dirty"].str.strip().apply(lambda x: f" {x} ")
df.loc[sample_idx[4:6],"Name_dirty"]="unknown"


In [12]:
sample_idx

Index([709, 439, 840, 720, 39, 290], dtype='int64')

In [13]:
df.loc[709]["Name_dirty"]

'MOUBAREK, MASTER. HALIM GONIOS ("WILLIAM GEORGE")'

**DATA QUALITY CHECKLIST**
Quick checklist
1.Basic Dataset Overview
2.Missing Values Summary
3.Duplicates
4.Data type Validation
5.Constant and quasi constant Columns
6.ID like columns
7.String inconsistencies
8.High Null columns
9.High zero Columns(for numeric features)

**BASIC DATASET OVERVIEW**


In [18]:
df.shape

(891, 14)

In [19]:
df.dtypes

PassengerId       int64
Survived          int64
Pclass            int64
Name                str
Sex                 str
Age             float64
SibSp             int64
Parch             int64
Ticket              str
Fare            float64
Cabin               str
Embarked            str
constant_col      int64
Name_dirty          str
dtype: object

In [20]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'constant_col',
       'Name_dirty'],
      dtype='str')

In [22]:
df.nunique()

PassengerId     891
Survived          2
Pclass            3
Name            891
Sex               2
Age              88
SibSp             7
Parch             7
Ticket          681
Fare            248
Cabin           147
Embarked          3
constant_col      1
Name_dirty      888
dtype: int64

**missing values Summary**

In [10]:
missing_count=df.isna().sum().sort_values(ascending=False)
missing_percent=(df.isna().mean()*100).sort_values(ascending=False)

missing_summary= pd.DataFrame(
    {"missing_count":missing_count,
    "missing_percent":missing_percent}
)
print("Missing Values Summary")
missing_summary

Missing Values Summary


,missing_count,missing_percent
Cabin,687,77.104377
Age,177,19.865320
Embarked,2,0.224467
PassengerId,0,0.000000
Name,0,0.000000
Pclass,0,0.000000
Survived,0,0.000000
Sex,0,0.000000
Parch,0,0.000000
SibSp,0,0.000000


**Duplicates**

In [9]:
df_duplicate=df.duplicated()
num_duplicates=df_duplicate.sum()
num_duplicates


np.int64(0)

**constant and quasi constant columns**

In [5]:
n_rows=len(df)
nunique=df.nunique()

const_cols=nunique[nunique==1].index.to_list()
print(const_cols)

[]


In [6]:
df.nunique()

PassengerId    891
Survived         2
Pclass           3
Name           891
Sex              2
Age             88
SibSp            7
Parch            7
Ticket         681
Fare           248
Cabin          147
Embarked         3
dtype: int64

In [7]:
quasi_constantcols=[]
for col in df.columns:
    top_freq=df[col].value_counts(normalize=True,dropna=False).values[0]
    if top_freq>0.75 and col not in const_cols:
        quasi_constantcols.append(col)

print(quasi_constantcols)

['Parch', 'Cabin']


**ID LIKE COLUMNS**

In [8]:
#columns in which number of unqiue values = total number of rows there might be exceptions 
n_rows=len(df)
id_like_columns=[]
for col in df.columns:
    if df[col].nunique(dropna=False)==n_rows:
        id_like_columns.append(col)

print("ID like columns: ",id_like_columns)


ID like columns:  ['PassengerId', 'Name']


**STRING INCONSISTENCIES**

In [16]:
object_cols=df.select_dtypes(include=["object","category"]).columns.to_list()
print("object based columns",object_cols)

#simple case:  strip spaces and convert it to lowercase
df["Name_clean"]=(
    df["Name_dirty"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace("unknown",np.nan)
                  
                  
                  
                  
                  )

object based columns ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked', 'Name_dirty']


C:\Users\beast\AppData\Local\Temp\ipykernel_12124\3721488866.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols=df.select_dtypes(include=["object","category"]).columns.to_list()


In [18]:
sample_idx
df.iloc[709]

PassengerId                                                  710
Survived                                                       1
Pclass                                                         3
Name           Moubarek, Master. Halim Gonios ("William George")
Sex                                                         male
Age                                                          NaN
SibSp                                                          1
Parch                                                          1
Ticket                                                      2661
Fare                                                     15.2458
Cabin                                                        NaN
Embarked                                                       C
Name_dirty     MOUBAREK, MASTER. HALIM GONIOS ("WILLIAM GEORGE")
Name_clean     moubarek, master. halim gonios ("william george")
Name: 709, dtype: object

In [19]:
high_null_threshold=0.4

high_null_columns=[missing_summary["missing_percent"]>=high_null_threshold*100]

print(high_null_columns)

[Cabin           True
Age            False
Embarked       False
PassengerId    False
Name           False
Pclass         False
Survived       False
Sex            False
Parch          False
SibSp          False
Fare           False
Ticket         False
Name: missing_percent, dtype: bool]
